In [1]:
from pathlib import Path
import pandas as pd

file_path = (
    Path("..")
    / "data"
    / "raw"
    / "performance"
    / "2019-09-18-subway-on-time-performance-v1.parquet"
)

df = pd.read_parquet(file_path)

df.head()

,stop_sequence,stop_id,parent_station,move_timestamp,stop_timestamp,travel_time_seconds,dwell_time_seconds,headway_trunk_seconds,headway_branch_seconds,service_date,...,trip_id,vehicle_label,vehicle_consist,direction,direction_destination,scheduled_arrival_time,scheduled_departure_time,scheduled_travel_time,scheduled_headway_branch,scheduled_headway_trunk
0,310,70107,place-lake,NaN,1.568794e+09,NaN,NaN,NaN,NaN,20190918,...,ADDED-1568746059,3870,3870,West,Boston College,23100.0,23100.0,180.0,NaN,NaN
1,310,70107,place-lake,NaN,1.568794e+09,NaN,NaN,NaN,NaN,20190918,...,ADDED-1568746058,3692,3692,West,Boston College,23100.0,23100.0,180.0,NaN,NaN
2,1,Alewife-02,place-alfcl,NaN,1.568794e+09,NaN,NaN,NaN,NaN,20190918,...,ADDED-1568746060,1872,1872|1873|1810|1811|1820|1821,South,Ashmont/Braintree,18960.0,18960.0,NaN,NaN,NaN
3,1,Alewife-01,place-alfcl,NaN,1.568794e+09,NaN,NaN,459.0,NaN,20190918,...,ADDED-1568746061,1867,1867|1866|1814|1815|1806|1807,South,Ashmont/Braintree,19440.0,19440.0,NaN,NaN,480.0
4,1,Braintree-01,place-brntn,NaN,1.568794e+09,NaN,NaN,NaN,NaN,20190918,...,ADDED-1568746062,1856,1856|1857|1879|1878|1819|1818,North,Alewife,18780.0,18780.0,NaN,NaN,NaN


In [2]:
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Rows: 44,970
Columns: 27


In [3]:
df.columns.tolist()

['stop_sequence',
 'stop_id',
 'parent_station',
 'move_timestamp',
 'stop_timestamp',
 'travel_time_seconds',
 'dwell_time_seconds',
 'headway_trunk_seconds',
 'headway_branch_seconds',
 'service_date',
 'route_id',
 'direction_id',
 'start_time',
 'vehicle_id',
 'branch_route_id',
 'trunk_route_id',
 'stop_count',
 'trip_id',
 'vehicle_label',
 'vehicle_consist',
 'direction',
 'direction_destination',
 'scheduled_arrival_time',
 'scheduled_departure_time',
 'scheduled_travel_time',
 'scheduled_headway_branch',
 'scheduled_headway_trunk']

In [4]:
df.iloc[0]

stop_sequence                            310
stop_id                                70107
parent_station                    place-lake
move_timestamp                           NaN
stop_timestamp                  1568793600.0
travel_time_seconds                      NaN
dwell_time_seconds                       NaN
headway_trunk_seconds                    NaN
headway_branch_seconds                   NaN
service_date                        20190918
route_id                             Green-B
direction_id                           False
start_time                             14400
vehicle_id                           G-10005
branch_route_id                      Green-B
trunk_route_id                         Green
stop_count                                 1
trip_id                     ADDED-1568746059
vehicle_label                           3870
vehicle_consist                         3870
direction                               West
direction_destination         Boston College
scheduled_

In [5]:
df.dtypes

stop_sequence                 int16
stop_id                         str
parent_station                  str
move_timestamp              float64
stop_timestamp              float64
travel_time_seconds         float64
dwell_time_seconds          float64
headway_trunk_seconds       float64
headway_branch_seconds      float64
service_date                  int64
route_id                        str
direction_id                   bool
start_time                    int64
vehicle_id                      str
branch_route_id                 str
trunk_route_id                  str
stop_count                    int16
trip_id                         str
vehicle_label                   str
vehicle_consist                 str
direction                       str
direction_destination           str
scheduled_arrival_time      float64
scheduled_departure_time    float64
scheduled_travel_time       float64
scheduled_headway_branch    float64
scheduled_headway_trunk     float64
dtype: object

In [6]:
df.isna().sum().sort_values(ascending=False)

headway_branch_seconds      14588
scheduled_headway_branch    12451
branch_route_id             11961
dwell_time_seconds           3818
headway_trunk_seconds        3603
scheduled_travel_time        2258
travel_time_seconds          1680
move_timestamp               1549
scheduled_headway_trunk       552
scheduled_departure_time      217
scheduled_arrival_time        217
stop_timestamp                126
vehicle_label                  11
route_id                        0
parent_station                  0
stop_sequence                   0
service_date                    0
stop_id                         0
start_time                      0
trip_id                         0
stop_count                      0
trunk_route_id                  0
vehicle_id                      0
direction_id                    0
direction_destination           0
vehicle_consist                 0
direction                       0
dtype: int64

In [7]:
missing = df.isna().sum()

missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": (missing / len(df) * 100).round(2)
})

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
].sort_values("missing_count", ascending=False)

missing_summary

,missing_count,missing_percent
headway_branch_seconds,14588,32.44
scheduled_headway_branch,12451,27.69
branch_route_id,11961,26.60
dwell_time_seconds,3818,8.49
headway_trunk_seconds,3603,8.01
scheduled_travel_time,2258,5.02
travel_time_seconds,1680,3.74
move_timestamp,1549,3.44
scheduled_headway_trunk,552,1.23
scheduled_departure_time,217,0.48


In [8]:
branch_missing = df[df["branch_route_id"].isna()]

branch_missing["route_id"].value_counts()

route_id
Orange      6019
Blue        4327
Mattapan    1605
Red           10
Name: count, dtype: int64

In [9]:
branch_route_profile = (
    df.groupby("route_id")["branch_route_id"]
    .agg(
        total_rows="size",
        missing_rows=lambda s: s.isna().sum(),
        non_missing_rows=lambda s: s.notna().sum()
    )
)

branch_route_profile["missing_percent"] = (
    branch_route_profile["missing_rows"]
    / branch_route_profile["total_rows"]
    * 100
).round(2)

branch_route_profile.sort_values("missing_percent", ascending=False)

,total_rows,missing_rows,non_missing_rows,missing_percent
route_id,,,,
Blue,4327,4327,0,100.00
Mattapan,1605,1605,0,100.00
Orange,6019,6019,0,100.00
Red,7600,10,7590,0.13
Green-D,5465,0,5465,0.00
Green-C,6293,0,6293,0.00
Green-B,7611,0,7611,0.00
Green-E,6050,0,6050,0.00


In [11]:
headway_missing_profile = (
    df.groupby("route_id")
    .agg(
        total_rows=("route_id", "size"),
        branch_headway_missing=(
            "headway_branch_seconds",
            lambda s: s.isna().sum()
        ),
        scheduled_branch_headway_missing=(
            "scheduled_headway_branch",
            lambda s: s.isna().sum()
        )
    )
)

headway_missing_profile["branch_headway_missing_percent"] = (
    headway_missing_profile["branch_headway_missing"]
    / headway_missing_profile["total_rows"] * 100
).round(2)

headway_missing_profile["scheduled_branch_headway_missing_percent"] = (
    headway_missing_profile["scheduled_branch_headway_missing"]
    / headway_missing_profile["total_rows"] * 100
).round(2)

headway_missing_profile.sort_values(
    "branch_headway_missing_percent",
    ascending=False
)

,total_rows,branch_headway_missing,scheduled_branch_headway_missing,branch_headway_missing_percent,scheduled_branch_headway_missing_percent
route_id,,,,,
Blue,4327,4327,4327,100.00,100.00
Mattapan,1605,1605,1605,100.00,100.00
Orange,6019,6019,6019,100.00,100.00
Red,7600,773,153,10.17,2.01
Green-D,5465,508,81,9.30,1.48
Green-C,6293,483,96,7.68,1.53
Green-E,6050,408,12,6.74,0.20
Green-B,7611,465,158,6.11,2.08


In [12]:
timestamp_missing_profile = (
    df[["move_timestamp", "stop_timestamp"]]
    .isna()
    .value_counts()
    .rename("row_count")
    .reset_index()
)

timestamp_missing_profile

,move_timestamp,stop_timestamp,row_count
0,False,False,43295
1,True,False,1549
2,False,True,126
